**Curated source notebook — archival code, not validated as end-to-end runnable.**\n\nExecution outputs and environment metadata were removed for repository hygiene. Original computational claims require independent validation; see `docs/AUDIT_STATUS.md`.\n

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import glob
import pandas as pd

#  Change this path to the folder where your TPM files are stored
data_folder = "/content/drive/MyDrive/GSE310675_TPM"

# --- 1) Collect all TPM files ---
# This finds every file that ends with "_Gene_TPM.txt.gz"
files = glob.glob(os.path.join(data_folder, "*_Gene_TPM.txt.gz"))

# Extract sample names (remove .txt.gz)
samples = [os.path.basename(f).replace(".txt.gz", "") for f in files]

# --- 2) Define the KRAS Q61H-mutant cell lines ---
# These are known from the GEO dataset description
q61h_lines = [
    "Hs766T",
    "HuPT3",
    "Pa14C",
    "T3M4"
]

# --- 3) Assign each sample to its correct group ---
# We detect the cell line name from the filename:
# Example: GSM9306524_Hs766T_Gene_TPM → extract "Hs766T"
metadata_rows = []

for s in samples:
    cell_line = s.split("_")[1]   # get the cell line name

    if cell_line in q61h_lines:
        group = "KRAS_Q61H"      # mutant group
    else:
        group = "Non_Q61H"       # control group

    metadata_rows.append([s, group])

# --- 4) Create the metadata DataFrame ---
metadata = pd.DataFrame(metadata_rows, columns=["sample", "group"])

# --- 5) Save to metadata.csv ---
out_path = os.path.join(data_folder, "metadata.csv")
metadata.to_csv(out_path, index=False)

print("✅ metadata.csv created successfully!")
print("File saved at:", out_path)
print("\nMetadata preview:")
print(metadata)


In [ ]:
import pandas as pd
import os, glob
from functools import reduce

data_folder = "/content/drive/MyDrive/GSE310675_TPM"

# 1) Load metadata
metadata = pd.read_csv(os.path.join(data_folder, "metadata.csv"))

# 2) Load TPM files
files = glob.glob(os.path.join(data_folder, "*_Gene_TPM.txt.gz"))

tpm_list = []

for f in files:
    sample_name = os.path.basename(f).replace(".txt.gz", "")

    df = pd.read_csv(f, sep="\t", compression="gzip")

    gene_col = df.columns[0]
    tpm_col  = df.columns[1]

    df = df[[gene_col, tpm_col]]
    df.rename(columns={gene_col: "gene", tpm_col: sample_name}, inplace=True)

    tpm_list.append(df)

# 3) Merge all TPM tables
tpm = reduce(lambda left, right: pd.merge(left, right, on="gene"), tpm_list)
tpm = tpm.set_index("gene")

print("TPM matrix shape:", tpm.shape)
tpm.head()


In [ ]:
# 4) Save TPM matrix as CSV
output_path = os.path.join(data_folder, "TPM_matrix.csv")
tpm.to_csv(output_path)

print("✅ TPM matrix saved successfully!")
print("File location:", output_path)


In [ ]:
import numpy as np
import pandas as pd

# log2(TPM + 1)
log_tpm = np.log2(tpm + 1)

print("log_tpm shape:", log_tpm.shape)
log_tpm.head()


In [ ]:
# Load metadata again (just to be safe)
metadata_path = "/content/drive/MyDrive/GSE310675_TPM/metadata.csv"
metadata = pd.read_csv(metadata_path)

# Extract sample names for each group
group_q61h = metadata[metadata["group"] == "KRAS_Q61H"]["sample"].tolist()
group_non  = metadata[metadata["group"] == "Non_Q61H"]["sample"].tolist()

print("KRAS_Q61H samples:", group_q61h)
print("Non_Q61H samples:", group_non)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,6))
log_tpm.boxplot(rot=90)
plt.title("QC: Boxplot of log2(TPM + 1)")
plt.ylabel("log2(TPM + 1)")
plt.show()


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# data of PCA
scaled = StandardScaler().fit_transform(log_tpm.T)
pca = PCA(n_components=2)
pc = pca.fit_transform(scaled)

pca_df = pd.DataFrame({
    "PC1": pc[:,0],
    "PC2": pc[:,1],
    "group": metadata["group"].values,
    "sample": metadata["sample"].values
})

plt.figure(figsize=(8,6))
for g in pca_df["group"].unique():
    subset = pca_df[pca_df["group"] == g]
    plt.scatter(subset["PC1"], subset["PC2"], s=100, label=g)

for i,row in pca_df.iterrows():
    plt.text(row["PC1"]+0.05, row["PC2"]+0.05, row["sample"], fontsize=8)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.title("QC: PCA Plot")
plt.legend()
plt.show()


In [ ]:
import seaborn as sns
import numpy as np


top_genes = log_tpm.var(axis=1).sort_values(ascending=False).head(50).index
heat_data = log_tpm.loc[top_genes]

plt.figure(figsize=(10,12))
sns.heatmap(heat_data, cmap="viridis")
plt.title("QC: Heatmap of Top 50 High-Variance Genes")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df = deg.copy()

# Compute -log10(p-value)
df["neglog10p"] = -np.log10(df["pval"] + 1e-300)

# Thresholds (raw p-value)
fc_cut = 1
p_cut = 0.01

# Colors
df["color"] = "gray"
df.loc[(df["logFC"] > fc_cut) & (df["pval"] < p_cut), "color"] = "red"
df.loc[(df["logFC"] < -fc_cut) & (df["pval"] < p_cut), "color"] = "blue"

plt.figure(figsize=(10,7))

# Scatter
plt.scatter(df["logFC"], df["neglog10p"], c=df["color"], s=25, alpha=0.7)

# Threshold lines
plt.axvline(fc_cut, linestyle="--", color="black")
plt.axvline(-fc_cut, linestyle="--", color="black")
plt.axhline(-np.log10(p_cut), linestyle="--", color="black")

# Label top genes
top_up = df[(df["logFC"] > fc_cut)].nlargest(5, "neglog10p")
top_down = df[(df["logFC"] < -fc_cut)].nlargest(5, "neglog10p")

for _, row in top_up.iterrows():
    plt.text(row["logFC"]+0.1, row["neglog10p"]+0.1, row["gene"], color="red")

for _, row in top_down.iterrows():
    plt.text(row["logFC"]-0.3, row["neglog10p"]+0.1, row["gene"], color="blue")

plt.xlabel("log2 Fold Change")
plt.ylabel("-log10(p-value)")
plt.title("Volcano Plot (Raw p-value, KRAS_Q61H vs Non_Q61H)")
plt.grid(alpha=0.2)
plt.show()


In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import t
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt

# ==========================================================
# 1) Load TPM + metadata
# ==========================================================
data_folder = "/content/drive/MyDrive/GSE310675_TPM"

tpm = pd.read_csv(os.path.join(data_folder, "TPM_matrix.csv"))
tpm = tpm.set_index("gene")

metadata = pd.read_csv(os.path.join(data_folder, "metadata.csv"))

group_q61h = metadata[metadata["group"] == "KRAS_Q61H"]["sample"].tolist()
group_non  = metadata[metadata["group"] == "Non_Q61H"]["sample"].tolist()

# ==========================================================
# 2) log2(TPM + 1)
# ==========================================================
log_tpm = np.log2(tpm + 1)

# group matrices
X = log_tpm[group_q61h].values
Y = log_tpm[group_non].values

# ==========================================================
# 3) limma-like Moderated t-test
# ==========================================================

# means
mx = X.mean(axis=1)
my = Y.mean(axis=1)
logFC = mx - my

# variances
vx = X.var(axis=1, ddof=1)
vy = Y.var(axis=1, ddof=1)

nx = X.shape[1]
ny = Y.shape[1]

# classical pooled variance
s2 = ((nx-1)*vx + (ny-1)*vy) / (nx + ny - 2)

# Empirical Bayes shrinkage (limma style)
s2_0 = np.median(s2)      # prior variance
df0  = 5                  # prior degrees of freedom

# posterior variance
s2_post = (df0*s2_0 + (nx + ny - 2)*s2) / (df0 + nx + ny - 2)

# moderated t-statistic
t_mod = logFC / np.sqrt(s2_post * (1/nx + 1/ny))
df_mod = df0 + nx + ny - 2

# p-values
pvals = 2 * (1 - t.cdf(np.abs(t_mod), df=df_mod))

# FDR correction
FDR = multipletests(pvals, method='fdr_bh')[1]

# ==========================================================
# 4) Build DEG table
# ==========================================================
deg = pd.DataFrame({
    "gene": log_tpm.index,
    "logFC": logFC,
    "pval": pvals,
    "FDR": FDR,
    "t_mod": t_mod
}).sort_values("pval")

# Save cleaned table
out_path = os.path.join(data_folder, "DEG_limma_like.csv")
deg.to_csv(out_path, index=False)
print("✅ DEG table saved to:", out_path)

# Top genes
print("\n🔥 Top 10 Upregulated:")
print(deg.sort_values("logFC", ascending=False).head(5)[["gene","logFC","pval","FDR"]])

print("\n❄️ Top 10 Downregulated:")
print(deg.sort_values("logFC", ascending=True).head(5)[["gene","logFC","pval","FDR"]])

# ==========================================================
# 5) Volcano Plot (limma-like)
# ==========================================================
df = deg.copy()
df["neglog10FDR"] = -np.log10(df["FDR"] + 1e-300)

fc_cut = 1
fdr_cut = 0.1   # relaxed for small sample sizes

df["color"] = "gray"
df.loc[(df["logFC"] > fc_cut) & (df["FDR"] < fdr_cut), "color"] = "red"
df.loc[(df["logFC"] < -fc_cut) & (df["FDR"] < fdr_cut), "color"] = "blue"

plt.figure(figsize=(10,7))
plt.scatter(df["logFC"], df["neglog10FDR"], c=df["color"], alpha=0.7, s=25)

plt.axvline(fc_cut, linestyle="--", color="black")
plt.axvline(-fc_cut, linestyle="--", color="black")
plt.axhline(-np.log10(fdr_cut), linestyle="--", color="black")

# Label top genes
top_up = df[(df["logFC"] > fc_cut) & (df["FDR"] < fdr_cut)].nlargest(5, "neglog10FDR")
top_down = df[(df["logFC"] < -fc_cut) & (df["FDR"] < fdr_cut)].nlargest(5, "neglog10FDR")

for _, row in top_up.iterrows():
    plt.text(row["logFC"]+0.1, row["neglog10FDR"]+0.1, row["gene"], color="red")

for _, row in top_down.iterrows():
    plt.text(row["logFC"]-0.3, row["neglog10FDR"]+0.1, row["gene"], color="blue")

plt.xlabel("log2 Fold Change")
plt.ylabel("-log10(FDR)")
plt.title("Volcano Plot (limma-like moderated t-test)")
plt.grid(alpha=0.2)
plt.show()


In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

# --------- 1) Paths ---------
data_folder = "/content/drive/MyDrive/GSE310675_TPM"

tpm_path      = os.path.join(data_folder, "TPM_matrix.csv")
metadata_path = os.path.join(data_folder, "metadata.csv")

# --------- 2) Load TPM matrix ---------
# TPM_matrix.csv should have a 'gene' column as the first column
tpm = pd.read_csv(tpm_path)
tpm = tpm.set_index("gene")

print("TPM shape:", tpm.shape)

# --------- 3) Load metadata and get sample groups ---------
metadata = pd.read_csv(metadata_path)

# metadata must contain: 'sample' and 'group' columns
group_q61h = metadata[metadata["group"] == "KRAS_Q61H"]["sample"].tolist()
group_non  = metadata[metadata["group"] == "Non_Q61H"]["sample"].tolist()

print("KRAS_Q61H samples:", group_q61h)
print("Non_Q61H samples:", group_non)

# تأكيد إن كل الأعمدة دي موجودة فعلاً في TPM
missing_q61h = [s for s in group_q61h if s not in tpm.columns]
missing_non  = [s for s in group_non  if s not in tpm.columns]

if missing_q61h or missing_non:
    print("⚠ WARNING: Some samples in metadata are not found in TPM_matrix columns!")
    print("Missing in Q61H:", missing_q61h)
    print("Missing in Non_Q61H:", missing_non)

# --------- 4) log2(TPM + 1) transformation ---------
log_tpm = np.log2(tpm + 1)

# --------- 5) Prepare group matrices for t-test ---------
X = log_tpm[group_q61h].values  # shape: (n_genes, n_q61h_samples)
Y = log_tpm[group_non].values   # shape: (n_genes, n_non_samples)

# Welch's t-test for each gene (axis=1 over samples)
tvals, pvals = ttest_ind(X, Y, axis=1, equal_var=False, nan_policy='omit')

# FDR correction (Benjamini-Hochberg)
fdr = multipletests(pvals, method='fdr_bh')[1]

# --------- 6) Build DEG table ---------
mean_q61h = log_tpm[group_q61h].mean(axis=1)
mean_non  = log_tpm[group_non].mean(axis=1)

deg = pd.DataFrame({
    "gene": log_tpm.index,
    "logFC": mean_q61h - mean_non,   # Q61H - Non_Q61H
    "pval": pvals,
    "FDR": fdr
}).sort_values("logFC", ascending=False)

# --------- 7) Filter significant DEGs ---------
deg_sig_up   = deg[(deg["logFC"] >= 1) & (deg["FDR"] < 0.05)]
deg_sig_down = deg[(deg["logFC"] <= -1) & (deg["FDR"] < 0.05)]

print("Significant upregulated genes:", deg_sig_up.shape[0])
print("Significant downregulated genes:", deg_sig_down.shape[0])

# --------- 8) Save all results ---------
out_path = os.path.join(data_folder, "DEG_TPM_ttest_FDR.csv")
deg.to_csv(out_path, index=False)
print("✅ DEG table saved to:", out_path)

# --------- 9) Top 5 up / down ---------
top5_up   = deg.sort_values("logFC", ascending=False).head(5)
top5_down = deg.sort_values("logFC", ascending=True).head(5)

print("\n🔥 Top 5 Upregulated in KRAS_Q61H:")
print(top5_up)

print("\n❄️ Top 5 Downregulated in KRAS_Q61H:")
print(top5_down)


In [ ]:
pip install plotnine


In [ ]:
from plotnine import *
import pandas as pd

# تأكد أن اسم الـ index موجود
log_tpm.index.name = "Gene"

# تحويل wide → long
data_long = (
    log_tpm
    .reset_index()
    .melt(id_vars="Gene",
          var_name="Sample",
          value_name="Expression")
)

(
    ggplot(data_long, aes(x='Sample', y='Expression', fill='Sample')) +
    geom_boxplot(outlier_size=0.5) +
    theme_bw() +
    theme(
        axis_text_x=element_text(rotation=90, hjust=1),
        legend_position='none'
    ) +
    labs(
        title="QC: Boxplot of log2(TPM + 1)",
        y="log2(TPM + 1)",
        x="Sample"
    )
)


In [ ]:
(
    ggplot(pca_df, aes('PC1', 'PC2', label='sample')) +
    geom_point(size=4, color='steelblue') +
    geom_text(nudge_y=0.2, size=8) +
    theme_bw() +
    labs(
        title='PCA of log2(TPM + 1)',
        x=f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)",
        y=f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)"
    )
)


In [ ]:
############################################################
# 1) Mount Google Drive in R-Colab
############################################################
system("pip install -q google-colab")  # ضروري لضمان توفر الحزمة
library(rstudioapi)

# تشغيل mount عبر Python داخل R
system("python3 - << 'EOF'\nfrom google.colab import drive\ndrive.mount('/content/drive')\nEOF")

############################################################
# 2) Load TPM file from Google Drive
############################################################
tpm <- read.csv("/content/drive/MyDrive/GSE310675_TPM/TPM_matrix.csv",
                header = TRUE, stringsAsFactors = FALSE)

# Set gene names as rownames
rownames(tpm) <- tpm$gene
tpm$gene <- NULL

# log2(TPM + 1)
log_tpm <- log2(tpm + 1)

############################################################
# 3) BOXPlot using ggplot2
############################################################
library(tidyr)
library(ggplot2)
library(dplyr)

log_tpm_df <- as.data.frame(log_tpm)
log_tpm_df$Gene <- rownames(log_tpm_df)

data_long <- log_tpm_df %>%
  pivot_longer(cols = -Gene, names_to = "Sample", values_to = "Expression")

p1 <- ggplot(data_long, aes(Sample, Expression, fill = Sample)) +
  geom_boxplot(outlier.size = 0.4) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 90, hjust = 1),
        legend.position = "none") +
  labs(title = "QC: Boxplot of log2(TPM + 1)",
       y = "log2(TPM + 1)")

print(p1)



enrichment

In [ ]:
# Install required packages (run once)
!pip install --quiet gprofiler-official gseapy pandas matplotlib seaborn

# Imports
import os
import pandas as pd
import numpy as np
from gprofiler import GProfiler
import gseapy as gp
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set(style="whitegrid")


In [ ]:
# --- Option A: use the example top lists you provided ---
up_genes = ["LDOC1","LDHB","TIMP3","NMNAT2","KRT17"]
down_genes = ["CEACAM6","HOXB2","HOXB8","AMH","HOXB6"]

# --- Option B (recommended): upload your full DEG table (CSV/TSV) if you have it ---
# If you want to upload a file from your local machine:
# from google.colab import files
# uploaded = files.upload()   # then choose file
# df = pd.read_csv(list(uploaded.keys())[0])   # adjust delimiter if needed

# Example: if you uploaded a CSV with columns: gene, logFC, pval, FDR
# you can create ranked list for preranked GSEA:
# df['Gene'] = df['gene_symbol_column']
# prerank_series = df.set_index('Gene')['logFC'].sort_values(ascending=False)


In [ ]:
gp_client = GProfiler(return_dataframe=True)

def run_gprofiler(gene_list, organism='hsapiens', sources=['GO:BP','KEGG','REAC']):
    """
    Run g:Profiler over-representation analysis.
    sources: list of sources to query; 'GO:BP','KEGG','REAC' recommended.
    Returns a dataframe of results.
    """
    res = gp_client.profile(organism=organism,
                            query=gene_list,
                            sources=sources,
                            user_threshold=0.05,        # p-value threshold
                            significance_threshold_method='fdr')  # BH
    return res

# Run for up- and down- genes
res_up = run_gprofiler(up_genes)
res_down = run_gprofiler(down_genes)

# Save results
os.makedirs("enrichment_results", exist_ok=True)
res_up.to_csv("enrichment_results/gprofiler_up.csv", index=False)
res_down.to_csv("enrichment_results/gprofiler_down.csv", index=False)

# Quick preview
print("Upregulated results (top):")
display(res_up[['source','native','name','p_value','effective_domain_size','term_size','query_size','intersection_size']].head(10))

print("Downregulated results (top):")
display(res_down[['source','native','name','p_value','effective_domain_size','term_size','query_size','intersection_size']].head(10))


In [ ]:
# Valid databases for Enrichr
enrichr_dbs = [
    'GO_Biological_Process_2021',
    'KEGG_2021_Human',
    'Reactome_2022' # Changed from Reactome_2024 to Reactome_2022
]

def run_enrichr(gene_list, gene_set_libs=enrichr_dbs, cutoff=0.05):
    enr = gp.enrichr(
        gene_list=gene_list,
        gene_sets=gene_set_libs,
        organism='Human',
        outdir=None,
        cutoff=cutoff
    )
    return enr.results

# Run
enr_up = run_enrichr(up_genes)
enr_down = run_enrichr(down_genes)

print("Enrichr (GO BP) top for upregulated:")
# Correctly filter the DataFrame by the 'Gene_set' column
go_bp_results = enr_up[enr_up['Gene_set'] == 'GO_Biological_Process_2021']
display(go_bp_results.head(10))


In [ ]:
def barplot_top_terms(gprofiler_df, top_n=10, title="Top enriched terms", savepath=None):
    # Keep only significant (p_value already adjusted by our call's 'fdr' method)
    df = gprofiler_df.copy()
    df = df[df['p_value'] <= 0.05].copy()
    if df.shape[0]==0:
        print("No significant terms at adjusted p <= 0.05.")
        return
    # Take top_n by p-value
    df = df.sort_values('p_value').head(top_n)[::-1]  # reverse for plotting
    df['-log10(padj)'] = -np.log10(df['p_value'])
    plt.figure(figsize=(8, max(3, 0.5*len(df))))
    sns.barplot(x='-log10(padj)', y='name', data=df, edgecolor="k")
    plt.xlabel("-log10(adjusted p-value)")
    plt.ylabel("")
    plt.title(title)
    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=300)
    plt.show()

# Example plots
barplot_top_terms(res_up, top_n=10, title="g:Profiler: Upregulated (Top terms)", savepath="enrichment_results/gprofiler_up_top.png")
barplot_top_terms(res_down, top_n=10, title="g:Profiler: Downregulated (Top terms)", savepath="enrichment_results/gprofiler_down_top.png")


In [ ]:
def dotplot_terms(gprofiler_df, top_n=15, title="Dotplot of enriched terms", savepath=None):
    df = gprofiler_df.copy()
    df = df[df['p_value'] <= 0.05].copy()
    if df.shape[0]==0:
        print("No significant terms at adjusted p <= 0.05.")
        return
    df = df.sort_values('p_value').head(top_n)
    df['-log10(padj)'] = -np.log10(df['p_value'])
    plt.figure(figsize=(8, max(4, 0.4*len(df))))
    sns.scatterplot(x='intersection_size', y='name', size='term_size', hue='-log10(padj)',
                    data=df, legend='brief', sizes=(50, 400))
    plt.xlabel("Genes in your list (intersection size)")
    plt.ylabel("")
    plt.title(title)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=300)
    plt.show()

# Example dotplots
dotplot_terms(res_up, top_n=12, title="g:Profiler Dotplot - Upregulated", savepath="enrichment_results/gprofiler_up_dotplot.png")
dotplot_terms(res_down, top_n=12, title="g:Profiler Dotplot - Downregulated", savepath="enrichment_results/gprofiler_down_dotplot.png")


In [ ]:
# Example: create a preranked series from uploaded CSV (must have gene symbol and logFC)
# df = pd.read_csv("your_full_deg_table.csv")
# Ensure columns: 'gene' and 'logFC' (adjust names below)
# prerank_df = df[['gene_symbol_col','logFC']].dropna()
# prerank_df = prerank_df.groupby('gene_symbol_col')['logFC'].mean().reset_index()
# prerank_df = prerank_df.sort_values('logFC', ascending=False)
# prerank_path = "preranked.rnk"
# prerank_df.to_csv(prerank_path, sep="\t", index=False, header=False)

# Run preranked GSEA (example with KEGG or Hallmark)
# gsea_res = gp.prerank(rnk=prerank_path, gene_sets='KEGG_2019_Human', outdir='gsea_out', permutation_num=100)
# print(gsea_res.res2d.head())


In [ ]:
# Files already written into enrichment_results/
!ls -lah enrichment_results


In [ ]:
def plot_enrichment(df, title):
    df = df.sort_values("p_value").head(12)
    df["-log10(p)"] = -np.log10(df["p_value"])

    plt.figure(figsize=(8,6))
    sns.barplot(y="name", x="-log10(p)", data=df, palette="viridis")
    plt.title(title)
    plt.xlabel("-log10(FDR)")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

plot_enrichment(res_up, "Top Enriched Terms (Upregulated)")
plot_enrichment(res_down, "Top Enriched Terms (Downregulated)")


In [ ]:
!pip install --quiet gseapy
import gseapy as gp # Added import statement for gseapy

# Valid databases for Enrichr
enrichr_dbs = [
    'GO_Biological_Process_2021',
    'KEGG_2021_Human',
    'Reactome_2022'
]

def run_enrichr(gene_list, gene_set_libs=enrichr_dbs, cutoff=0.05):
    enr = gp.enrichr(
        gene_list=gene_list,
        gene_sets=gene_set_libs,
        organism='Human',
        outdir=None,
        cutoff=cutoff
    )
    return enr.results

# Run enrichment
enr_up = run_enrichr(up_genes)
enr_down = run_enrichr(down_genes)

# Display top KEGG pathways for upregulated genes
print("Enrichr (KEGG 2021) top for upregulated:")
kegg_up_results = enr_up[enr_up['Gene_set'] == 'KEGG_2021_Human']
display(kegg_up_results.head(10))

# Optional: Display top KEGG pathways for downregulated genes
print("Enrichr (KEGG 2021) top for downregulated:")
kegg_down_results = enr_down[enr_down['Gene_set'] == 'KEGG_2021_Human']
display(kegg_down_results.head(10))
! mv /content/enrichment_results

In [ ]:
# Run enrichment
enr_up = run_enrichr(up_genes)
enr_down = run_enrichr(down_genes)

# Display top Reactome 2022 pathways for upregulated genes
print("Enrichr (Reactome 2022) top for upregulated:")
reactome_up_results = enr_up[enr_up['Gene_set'] == 'Reactome_2022']
display(reactome_up_results.head(10))

# Display top Reactome 2022 pathways for downregulated genes
print("Enrichr (Reactome 2022) top for downregulated:")
reactome_down_results = enr_down[enr_down['Gene_set'] == 'Reactome_2022']
display(reactome_down_results.head(10))


In [ ]:
# Files already written into enrichment_results/
!ls -lah {output_folder}